<h1>Chapter 6 - Planning & Reflection</h1>
<i>More Autonomy for your `TinyAgent`</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 6 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b &

# ▂▂▂▂▂▂▂▂▂▂▂▂

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [ ]:
import os
from illustrated_agents.llm import LLM

# Ollama
llm = LLM(model="ollama/gemma3:12b")

# Llama.cpp server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M", api_base="http://localhost:8080", api_key="sk-no-key-required")

# Llama-cpp-python server
# llm = LLM(model="openai/gemma-3-12b-it-Q4_K_M.gguf", api_base="http://localhost:8000/v1/", api_key="sk-no-key-required")

# LM Studio
# llm = LLM(model="lm_studio/gemma-3-12b-it", api_base="http://localhost:1234/v1", api_key="sk-no-key-required")

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_GEMINI_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash")
# llm = LLM(model="gemini/gemma-3-12b-it")

## 2 - Reflection

The great thing about the ReAct-based `TinyAgent` we created in Chapter 6 is that it allows us to add steps in between and reflection specifically. We covered many different types of reflection in the book and here we are going to show you how to implement a straightforward one, namely reflecting every N steps. Compared to the ones we covered in the book, it is much simpler and does not require complex coding. 

Let's start by creating our main class for reflecting:

In [ ]:
class Reflector:
    """Reflection module for agent self-evaluation."""

    def __init__(self, interval: int = 3):
        """Initialize the reflector.

        Arguments:
            interval: Reflect every N steps.
        """
        self.interval = interval

    @property
    def prompt(self) -> str:
        return """
REFLECTION: Take a moment to evaluate your progress.
- Are you making progress toward the goal?
- Should you try a different approach?
- What have you learned from previous steps?

Continue with your next THOUGHT and ACTION."
"""

    def should_reflect(self, step: int) -> bool:
        """Determine if the agent should reflect at this step."""
        return step > 0 and step % self.interval == 0


In this class, there are three actions that are of interest to this reflection module:

In [ ]:
from illustrated_agents.chapters.ch6_reflection import reflector_annotated; reflector_annotated

In practice, and as covered in the book, reflection can take many forms. It can be done during reasoning but also like we did above by explicitly prompting the model to do so. 

## 3 - Updating `agent.py`

Compared to the `ReAct` example in the previous notebook, adding a reflection is a breeze and only requires a couple lines of code. 

In [ ]:
from illustrated_agents import LLM, Memory, Tools, ReAct


class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: LLM, memory: Memory, tools: Tools, planner: ReAct, reflector: Reflector):
        self.llm = llm
        self.memory = memory
        self.tools = tools
        self.planner = planner
        self.reflector = reflector
        self.skills = None  # Chapter 6: Add Skills

        # Build system prompt with all components
        system_prompt = "You are a helpful AI agent.\n\n"
        system_prompt += self.planner.prompt + "\n\n"
        system_prompt += self.tools.prompt
        self.memory.add("user", system_prompt)

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task)

        # `Autonomy` loop
        for step in range(self.planner.max_steps):
            # Reflection step before taking the next action
            if self.reflector.should_reflect(step):
                self.memory.add("user", self.reflector.prompt)

            # Perform a step and check for completion
            result = self._step()
            if result is not None:
                return result

        return "Max steps reached without completion."

    def _step(self) -> str | None:
        """Perform a single step."""
        # Generate response and add to memory
        response = self.llm.generate(self.memory.get_messages())
        self.memory.add("assistant", response)

        # Parse planner's response to extract action if needed
        response = self.planner.parse(response)

        # Tool parsing and execution
        if self.tools.has_tool_call(response):
            return self._execute_action(response)

        return None

    def _execute_action(self, action: str) -> str | None:
        """Execute a tool action."""
        tool_call = self.tools.parse_tool_call(action)

        # Final answer ends the loop
        if tool_call["tool"] == "final_answer":
            return tool_call.get("kwargs", "")

        # Execute tool and extract the observation
        observation = self.tools.run_tool(tool_call)
        obs_prompt = f"OBSERVATION: {action} -> {observation}"
        self.memory.add("user", obs_prompt)

        return None

Since our `agent.py` is getting bigger, it becomes harder to see what we are adding. As always, let's showcase the diff to see these minimal changes:

In [ ]:
from illustrated_agents.chapters.ch6_reflection import tinyagents_diff; tinyagents_diff

As shown, we are merely adding the reflection prompt so that your `TinyAgent` is forced to start reflecting before continuing with the regular **THOUGHT**/**ACTION** steps.

Let's see what happens when we put this to the test!

In [ ]:
from illustrated_agents import Tools, Memory, ReAct


# Define tools
def calculator(a: str, b: str) -> float:
    return float(a) + float(b)

def get_weather(location: str) -> str:
    return f"Weather in {location}: Sunny, 72°F"

# Register tools
tools = Tools()
tools.add_tool("calculator", calculator, "Adds two numbers: calculator(a: str, b: str)")
tools.add_tool("get_weather", get_weather, "Gets weather: get_weather(location: str)")

# Memory
memory = Memory()

# ReAct
react = ReAct(max_steps=10)

# Reflector
reflector = Reflector(interval=3)

# Create agent
agent = TinyAgent(llm=llm, tools=tools, memory=memory, planner=react, reflector=reflector)

We run the same task as before but this time your `TinyAgent` will reflect every three steps.

In [ ]:
# Multi-step task with reasoning
agent.run("""
I'm planning a trip! Help me with these tasks:
1. What's the weather in New York City?
2. What's the weather in Los Angeles?
3. I saved $150.50 and my friend is giving me $75.25. How much do I have for the trip?

Based on the weather, which city would you recommend I visit?
""")

In [ ]:
agent.memory.get_messages()

Note how it reflects right before giving back the final output which pushes it to think a bit longer before giving back the final answer. This is a bit of an artificial example considering it is hard to say whether your `TinyAgent` should reflect every 1, 2, 3 or more steps. As covered in the book, there are many ways in which you can force a moment of self-reflection, even by using reflection as a tool!

# ▂▂▂▂▂▂▂▂▂▂▂▂